# Prepare Tromsø Radar Archives

Inventories the original EISCAT Tromsø bundles, splits them into 24 hourly archives, and records checksums and completion metadata. It does not perform detection or inference.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


## 1. Set repository paths and the campaign boundary


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timedelta, timezone
from collections import defaultdict
import copy
import hashlib
import json
import os
import re
import shutil
import subprocess
import tarfile

import pandas as pd

BASE = PROJECT_ROOT
TROMSO_ROOT = BASE / "data" / "tromso"
RAW_DIR = TROMSO_ROOT / "raw_download_bundles"
HOURLY_DIR = TROMSO_ROOT / "hourly_archives"
MANIFEST_DIR = TROMSO_ROOT / "archive_manifests"
COMPLETE_MARKER = MANIFEST_DIR / "PREPARATION_COMPLETE.json"

WORK_ROOT = Path(".cache/debris_tromso_archive_preparation")
CACHE_PATH = MANIFEST_DIR / "raw_bundle_inventory_cache.json"
INVENTORY_CSV = MANIFEST_DIR / "raw_bundle_inventory.csv"

for directory in [RAW_DIR, HOURLY_DIR, MANIFEST_DIR, WORK_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

CAMPAIGN_START = datetime(2018, 1, 4, 12, tzinfo=timezone.utc)
EXPECTED_HOURS = [
    (CAMPAIGN_START + timedelta(hours=index)).strftime("%Y%m%d_%H")
    for index in range(24)
]

                                                                                 
MAX_SOURCE_BUNDLES_PER_RUN = 1

                                                                                    
VERIFY_DRIVE_SHA256 = True

print("Original bundles:", RAW_DIR)
print("Verified hourly archives:", HOURLY_DIR)
print("Manifests:", MANIFEST_DIR)
print("Expected first/last hours:", EXPECTED_HOURS[0], EXPECTED_HOURS[-1])
print(f"Local free space: {shutil.disk_usage('.').free / 2**30:.1f} GiB")


## 2. Inventory the original TAR headers


In [ ]:
HOUR_PATTERN = re.compile(r"(?:^|/)(\d{8}_\d{2})(?:/|$)")


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, chunk_bytes=16 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_bytes)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def normalized_hour_member(member_name, hour_key):
    clean_name = member_name.replace("\\", "/").lstrip("./")
    parts = PurePosixPath(clean_name).parts
    if hour_key not in parts:
        raise RuntimeError(f"Hour {hour_key} is not a complete path component in {member_name!r}")
    index = parts.index(hour_key)
    relative = PurePosixPath(*parts[index:])
    if relative.is_absolute() or ".." in relative.parts:
        raise RuntimeError(f"Unsafe TAR member path: {member_name!r}")
    return relative.as_posix()


raw_archives = sorted(
    path for path in RAW_DIR.iterdir()
    if path.is_file() and path.name.strip().lower().endswith(".tar")
)
if not raw_archives:
    raise FileNotFoundError(f"No original TAR bundles found in {RAW_DIR}")

raw_state = [
    {"source_name": path.name, "source_size_bytes": path.stat().st_size}
    for path in raw_archives
]

cache = None
if CACHE_PATH.exists():
    try:
        candidate = json.loads(CACHE_PATH.read_text(encoding="utf-8"))
        if candidate.get("raw_state") == raw_state:
            cache = candidate
    except Exception as error:
        print("Ignoring unreadable inventory cache:", error)

if cache is None:
    inventory_rows = []
    for archive in raw_archives:
        print("Reading TAR headers:", archive.name)
        member_records = defaultdict(list)
        with tarfile.open(archive, "r:*") as source_tar:
            for member in source_tar:
                match = HOUR_PATTERN.search(member.name.replace("\\", "/"))
                if match and member.isfile():
                    hour_key = match.group(1)
                    normalized = normalized_hour_member(member.name, hour_key)
                    member_records[hour_key].append((normalized, int(member.size)))

        distinct_hours = len(member_records)
        for hour_key, records in sorted(member_records.items()):
            records = sorted(records)
            signature_text = "".join(f"{name}\0{size}\n" for name, size in records)
            inventory_rows.append({
                "hour_key": hour_key,
                "source_name": archive.name,
                "source_path": str(archive),
                "source_size_bytes": archive.stat().st_size,
                "source_distinct_hours": distinct_hours,
                "member_files": len(records),
                "member_payload_bytes": sum(size for _, size in records),
                "member_signature_sha256": hashlib.sha256(signature_text.encode()).hexdigest(),
            })

    cache = {
        "created_at_utc": utc_now(),
        "raw_state": raw_state,
        "inventory_rows": inventory_rows,
    }
    temp_cache = MANIFEST_DIR / "raw_bundle_inventory_cache.json.partial"
    temp_cache.write_text(json.dumps(cache, indent=2), encoding="utf-8")
    temp_cache.replace(CACHE_PATH)
else:
    inventory_rows = cache["inventory_rows"]
    print("Using cached header inventory from:", CACHE_PATH)

inventory = pd.DataFrame(inventory_rows).sort_values(
    ["hour_key", "source_distinct_hours", "source_name"]
).reset_index(drop=True)
inventory.to_csv(INVENTORY_CSV, index=False)
display(inventory.assign(
    source_gib=inventory["source_size_bytes"] / 2**30,
    member_payload_gib=inventory["member_payload_bytes"] / 2**30,
))

actual_hours = set(inventory["hour_key"])
expected_set = set(EXPECTED_HOURS)
missing_hours = sorted(expected_set - actual_hours)
extra_hours = sorted(actual_hours - expected_set)

print("Original TAR bundles:", len(raw_archives))
print("Unique internal UTC hours:", len(actual_hours))
print("Missing campaign hours:", missing_hours or "none")
print("Extra hours outside campaign:", extra_hours or "none")

if missing_hours:
    raise RuntimeError("The raw bundles do not contain all 24 expected campaign hours.")

canonical_rows = []
for hour_key in EXPECTED_HOURS:
    candidates = inventory[inventory["hour_key"] == hour_key].copy()
    if candidates["member_signature_sha256"].nunique() != 1:
        display(candidates)
        raise RuntimeError(
            f"Conflicting copies of {hour_key} exist. Their member lists/sizes differ; "
            "do not choose one by filename."
        )
                                                                     
    chosen = candidates.sort_values(
        ["source_distinct_hours", "source_size_bytes", "source_name"]
    ).iloc[0]
    canonical_rows.append(chosen.to_dict())

canonical = pd.DataFrame(canonical_rows).sort_values("hour_key").reset_index(drop=True)
assert canonical["hour_key"].tolist() == EXPECTED_HOURS

duplicate_count = int(len(inventory[inventory["hour_key"].isin(EXPECTED_HOURS)]) - 24)
print("Verified expected campaign hours: 24")
print("Identical duplicate hour copies ignored:", duplicate_count)
display(canonical[[
    "hour_key", "source_name", "source_distinct_hours", "member_files",
    "member_payload_bytes", "member_signature_sha256",
]])


## 3. Validate existing hourly outputs and choose the next source bundle


In [ ]:
def hourly_filename(hour_key):
    return f"eiscat_tromso_{hour_key}_uhf_l2_hdf5.tar"


def hour_manifest_path(hour_key):
    return MANIFEST_DIR / f"{hour_key}.json"


def existing_hour_is_verified(hour_key):
    archive_path = HOURLY_DIR / hourly_filename(hour_key)
    manifest_path = hour_manifest_path(hour_key)
    if not archive_path.exists() and not manifest_path.exists():
        return False
    if not archive_path.exists() or not manifest_path.exists():
        raise RuntimeError(
            f"Incomplete output state for {hour_key}: archive={archive_path.exists()}, "
            f"manifest={manifest_path.exists()}. Inspect it before rerunning."
        )
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("status") != "verified" or manifest.get("hour_key") != hour_key:
        raise RuntimeError(f"Invalid manifest for existing archive {hour_key}: {manifest_path}")
    if archive_path.stat().st_size != manifest.get("archive_size_bytes"):
        raise RuntimeError(f"Existing hourly archive size changed for {hour_key}")
    return True


verified_before = {
    hour_key for hour_key in EXPECTED_HOURS
    if existing_hour_is_verified(hour_key)
}

pending = canonical[~canonical["hour_key"].isin(verified_before)].copy()
pending_sources = list(dict.fromkeys(pending["source_name"].tolist()))
selected_sources = pending_sources[:MAX_SOURCE_BUNDLES_PER_RUN]

print(f"Already verified: {len(verified_before)} / 24")
print("Pending source bundles:", pending_sources or "none")
print("Selected this run:", selected_sources or "none")


## 4. Split one source bundle into normalized hourly TARs


In [ ]:
def run_live(command):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    completed = subprocess.run(command)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed ({completed.returncode}): {' '.join(command)}")


def verify_hourly_tar(path, expected_hour, expected_files, expected_payload_bytes):
    file_count = 0
    payload_bytes = 0
    seen_hours = set()
    seen_names = set()
    with tarfile.open(path, "r:*") as archive:
        for member in archive:
            if not member.isfile():
                continue
            match = HOUR_PATTERN.search(member.name.replace("\\", "/"))
            if not match:
                raise RuntimeError(f"Unscoped file in {path.name}: {member.name}")
            hour_key = match.group(1)
            normalized = normalized_hour_member(member.name, hour_key)
            if normalized != member.name:
                raise RuntimeError(f"Non-normalized member in {path.name}: {member.name}")
            if normalized in seen_names:
                raise RuntimeError(f"Duplicate member in {path.name}: {normalized}")
            seen_names.add(normalized)
            seen_hours.add(hour_key)
            file_count += 1
            payload_bytes += int(member.size)

    if seen_hours != {expected_hour}:
        raise RuntimeError(f"{path.name} contains hours {sorted(seen_hours)}, expected only {expected_hour}")
    if file_count != int(expected_files):
        raise RuntimeError(f"{path.name}: {file_count} files, expected {expected_files}")
    if payload_bytes != int(expected_payload_bytes):
        raise RuntimeError(
            f"{path.name}: {payload_bytes} payload bytes, expected {expected_payload_bytes}"
        )
    return {"member_files": file_count, "member_payload_bytes": payload_bytes}


def publish_hour(local_tar, record):
    hour_key = record["hour_key"]
    final_path = HOURLY_DIR / hourly_filename(hour_key)
    partial_path = HOURLY_DIR / f"{hourly_filename(hour_key)}.partial"
    manifest_path = hour_manifest_path(hour_key)

    if final_path.exists() or manifest_path.exists():
        raise RuntimeError(f"Refusing to overwrite existing output for {hour_key}")
    if partial_path.exists():
        print("Removing a prior incomplete partial copy:", partial_path.name)
        partial_path.unlink()

    local_validation = verify_hourly_tar(
        local_tar,
        hour_key,
        record["member_files"],
        record["member_payload_bytes"],
    )
    local_sha256 = sha256_file(local_tar)

    run_live(["rsync", "-ah", "--info=progress2", str(local_tar), str(partial_path)])
    if partial_path.stat().st_size != local_tar.stat().st_size:
        raise RuntimeError(f"output copy size mismatch for {hour_key}")

    drive_validation = verify_hourly_tar(
        partial_path,
        hour_key,
        record["member_files"],
        record["member_payload_bytes"],
    )
    drive_sha256 = sha256_file(partial_path) if VERIFY_DRIVE_SHA256 else None
    if drive_sha256 is not None and drive_sha256 != local_sha256:
        raise RuntimeError(f"output SHA-256 mismatch for {hour_key}")

    partial_path.replace(final_path)
    manifest = {
        "status": "verified",
        "hour_key": hour_key,
        "created_at_utc": utc_now(),
        "source_bundle": record["source_name"],
        "source_bundle_size_bytes": int(record["source_size_bytes"]),
        "source_member_signature_sha256": record["member_signature_sha256"],
        "archive_name": final_path.name,
        "archive_size_bytes": final_path.stat().st_size,
        "archive_sha256": local_sha256,
        "drive_sha256_verified": bool(VERIFY_DRIVE_SHA256),
        "member_files": local_validation["member_files"],
        "member_payload_bytes": local_validation["member_payload_bytes"],
        "normalized_top_level_directory": hour_key,
        "validation": {
            "local_tar_passed": True,
            "drive_tar_passed": True,
            "drive_size_passed": True,
            "drive_sha256_passed": bool(VERIFY_DRIVE_SHA256),
            "drive_member_files": drive_validation["member_files"],
        },
    }
    manifest_partial = MANIFEST_DIR / f"{hour_key}.json.partial"
    manifest_partial.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    manifest_partial.replace(manifest_path)
    print("VERIFIED:", hour_key, "→", final_path.name)


def split_source_bundle(source_name, records):
    source_path = RAW_DIR / source_name
    hour_records = {row["hour_key"]: row for row in records}
    if not hour_records:
        return

    required_bytes = int(sum(row["member_payload_bytes"] for row in records) * 1.03) + 8 * 2**30
    local_free = shutil.disk_usage(".").free
    if local_free < required_bytes:
        raise RuntimeError(
            f"Not enough local disk for {source_name}. Free={local_free/2**30:.1f} GiB; "
            f"required≈{required_bytes/2**30:.1f} GiB. Process on a runtime with more local disk."
        )

    safe_stem = re.sub(r"[^A-Za-z0-9._-]+", "_", source_path.stem.strip())
    bundle_work = WORK_ROOT / safe_stem
    if bundle_work.exists():
        shutil.rmtree(bundle_work)
    bundle_work.mkdir(parents=True)

    local_paths = {
        hour_key: bundle_work / hourly_filename(hour_key)
        for hour_key in hour_records
    }
    writers = {
        hour_key: tarfile.open(path, "w", format=tarfile.PAX_FORMAT)
        for hour_key, path in local_paths.items()
    }
    written_names = {hour_key: set() for hour_key in hour_records}
    written_files = defaultdict(int)
    written_bytes = defaultdict(int)

    print("Streaming source bundle once:", source_path.name)
    try:
        with tarfile.open(source_path, "r:*") as source_tar:
            for member in source_tar:
                match = HOUR_PATTERN.search(member.name.replace("\\", "/"))
                if not match or not member.isfile():
                    continue
                hour_key = match.group(1)
                if hour_key not in writers:
                    continue

                normalized = normalized_hour_member(member.name, hour_key)
                if normalized in written_names[hour_key]:
                    raise RuntimeError(f"Duplicate source member for {hour_key}: {normalized}")
                written_names[hour_key].add(normalized)

                payload = source_tar.extractfile(member)
                if payload is None:
                    raise RuntimeError(f"Could not read source member: {member.name}")
                clean_member = copy.copy(member)
                clean_member.name = normalized
                clean_member.uid = 0
                clean_member.gid = 0
                clean_member.uname = ""
                clean_member.gname = ""
                clean_member.pax_headers = {
                    key: value
                    for key, value in clean_member.pax_headers.items()
                    if key not in {"path", "linkpath"}
                }
                writers[hour_key].addfile(clean_member, payload)
                written_files[hour_key] += 1
                written_bytes[hour_key] += int(member.size)
    finally:
        for writer in writers.values():
            writer.close()

    for hour_key, record in hour_records.items():
        if written_files[hour_key] != int(record["member_files"]):
            raise RuntimeError(
                f"Streaming count mismatch for {hour_key}: "
                f"{written_files[hour_key]} vs {record['member_files']}"
            )
        if written_bytes[hour_key] != int(record["member_payload_bytes"]):
            raise RuntimeError(f"Streaming byte mismatch for {hour_key}")
        publish_hour(local_paths[hour_key], record)
        local_paths[hour_key].unlink()

    shutil.rmtree(bundle_work)


def publish_existing_single_hour_tar(record):
    hour_key = record["hour_key"]
    source_path = Path(record["source_path"])

    final_path = HOURLY_DIR / hourly_filename(hour_key)
    partial_path = HOURLY_DIR / f"{hourly_filename(hour_key)}.partial"
    manifest_path = hour_manifest_path(hour_key)

    if final_path.exists() or manifest_path.exists():
        raise RuntimeError(
            f"Refusing to overwrite an existing output for {hour_key}"
        )

    if partial_path.exists():
        print("Removing prior incomplete copy:", partial_path.name)
        partial_path.unlink()

                                                                           
                                                                              
    file_count = 0
    payload_bytes = 0
    hours_found = set()
    normalized_names = set()

    print("Validating existing single-hour TAR:", source_path.name)

    with tarfile.open(source_path, "r:*") as archive:
        for member in archive:
            if not member.isfile():
                continue

            match = HOUR_PATTERN.search(
                member.name.replace("\\", "/")
            )
            if not match:
                raise RuntimeError(
                    f"Unscoped file in {source_path.name}: {member.name}"
                )

            detected_hour = match.group(1)
            normalized = normalized_hour_member(
                member.name,
                detected_hour,
            )

            if normalized in normalized_names:
                raise RuntimeError(
                    f"Duplicate normalized member: {normalized}"
                )

            normalized_names.add(normalized)
            hours_found.add(detected_hour)
            file_count += 1
            payload_bytes += int(member.size)

    if hours_found != {hour_key}:
        raise RuntimeError(
            f"{source_path.name} contains {sorted(hours_found)}, "
            f"expected only {hour_key}"
        )

    if file_count != int(record["member_files"]):
        raise RuntimeError(
            f"{source_path.name} contains {file_count} files, "
            f"expected {record['member_files']}"
        )

    if payload_bytes != int(record["member_payload_bytes"]):
        raise RuntimeError(
            f"{source_path.name} contains {payload_bytes} payload bytes, "
            f"expected {record['member_payload_bytes']}"
        )

                                                                             
                                                
    print("Copying original TAR directly:", source_path.name)
    run_live([
        "rsync",
        "-ah",
        "--info=progress2",
        str(source_path),
        str(partial_path),
    ])

    if partial_path.stat().st_size != source_path.stat().st_size:
        raise RuntimeError(
            f"Copied TAR size mismatch for {hour_key}"
        )

    source_sha256 = sha256_file(source_path)

    if VERIFY_DRIVE_SHA256:
        copied_sha256 = sha256_file(partial_path)
        if copied_sha256 != source_sha256:
            raise RuntimeError(
                f"Copied TAR SHA-256 mismatch for {hour_key}"
            )
    else:
        copied_sha256 = None

    partial_path.replace(final_path)

    manifest = {
        "status": "verified",
        "hour_key": hour_key,
        "created_at_utc": utc_now(),
        "source_bundle": source_path.name,
        "source_bundle_size_bytes": source_path.stat().st_size,
        "source_distinct_hours": 1,
        "source_member_signature_sha256":
            record["member_signature_sha256"],
        "archive_name": final_path.name,
        "archive_size_bytes": final_path.stat().st_size,
        "archive_sha256": source_sha256,
        "drive_sha256_verified": bool(VERIFY_DRIVE_SHA256),
        "member_files": file_count,
        "member_payload_bytes": payload_bytes,
        "preparation_method": "direct_byte_for_byte_copy",
        "repacked": False,
        "validation": {
            "single_internal_hour_passed": True,
            "member_count_passed": True,
            "payload_bytes_passed": True,
            "copy_size_passed": True,
            "copy_sha256_passed": bool(VERIFY_DRIVE_SHA256),
        },
    }

    manifest_partial = MANIFEST_DIR / f"{hour_key}.json.partial"
    manifest_partial.write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )
    manifest_partial.replace(manifest_path)

    print(
        "VERIFIED DIRECT COPY:",
        hour_key,
        "→",
        final_path.name,
    )


for source_name in selected_sources:
    source_records = pending[
        pending["source_name"] == source_name
    ].to_dict("records")

    print("\n" + "=" * 90)
    print(
        "PREPARING",
        source_name,
        "→",
        [row["hour_key"] for row in source_records],
    )
    print("=" * 90)

                                                                      
                                                                  
    if (
        len(source_records) == 1
        and int(source_records[0]["source_distinct_hours"]) == 1
    ):
        publish_existing_single_hour_tar(source_records[0])
    else:
                                                        
        split_source_bundle(source_name, source_records)


## 5. Completion dashboard


In [ ]:
status_rows = []
for hour_key in EXPECTED_HOURS:
    archive_path = HOURLY_DIR / hourly_filename(hour_key)
    manifest_path = hour_manifest_path(hour_key)
    verified = existing_hour_is_verified(hour_key)
    status_rows.append({
        "hour_key": hour_key,
        "status": "verified" if verified else "pending",
        "archive_name": archive_path.name if archive_path.exists() else None,
        "archive_gib": archive_path.stat().st_size / 2**30 if archive_path.exists() else None,
        "manifest": str(manifest_path) if manifest_path.exists() else None,
    })

status = pd.DataFrame(status_rows)
display(status)
verified_count = int((status["status"] == "verified").sum())
print(f"VERIFIED HOURS: {verified_count} / 24")

if verified_count == 24:
    completion = {
        "status": "complete",
        "campaign": "EISCAT Tromsø UHF 2018-01-04 12 UTC to 2018-01-05 12 UTC",
        "created_at_utc": utc_now(),
        "expected_hours": EXPECTED_HOURS,
        "hourly_archive_directory": str(HOURLY_DIR),
        "hour_manifest_directory": str(MANIFEST_DIR),
        "archive_count": 24,
        "all_hour_manifests_verified": True,
    }
    marker_partial = MANIFEST_DIR / "PREPARATION_COMPLETE.json.partial"
    marker_partial.write_text(json.dumps(completion, indent=2), encoding="utf-8")
    marker_partial.replace(COMPLETE_MARKER)
    print("READY FOR GMF:", COMPLETE_MARKER)
else:
    if COMPLETE_MARKER.exists():
        raise RuntimeError(
            "A stale PREPARATION_COMPLETE.json exists even though fewer than 24 hours verify. "
            "Inspect the preparation folders before continuing."
        )
    print("Run the notebook again to prepare the next source bundle.")
